In [8]:
import pandas as pd

df = pd.read_csv("../data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
print(df.shape)
df.head()

(1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [9]:
df.info()
print(df.isnull().sum().sum())
print(df.duplicated().sum())
df["Attrition"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

Attrition
No     1233
Yes     237
Name: count, dtype: int64

In [10]:
# tidy names first
df.columns = df.columns.str.strip().str.lower()

# drop columns that never change (ignored if already gone)
df = df.drop(columns=["employeecount", "over18", "standardhours"], errors="ignore")

# 1/0 flag for attrition
df["attrition_flag"] = (df["attrition"] == "Yes").astype(int)

# groups for analysis
df["age_group"] = pd.cut(df["age"], bins=[17, 25, 35, 45, 55, 70],
                         labels=["18-25", "26-35", "36-45", "46-55", "56+"])
df["tenure_group"] = pd.cut(df["yearsatcompany"], bins=[-1, 2, 5, 10, 40],
                            labels=["0-2 yrs", "3-5 yrs", "6-10 yrs", "10+ yrs"])

df.to_csv("../data/attrition_clean.csv", index=False)
print(df.shape)
df.columns.tolist()

(1470, 35)


['age',
 'attrition',
 'businesstravel',
 'dailyrate',
 'department',
 'distancefromhome',
 'education',
 'educationfield',
 'employeenumber',
 'environmentsatisfaction',
 'gender',
 'hourlyrate',
 'jobinvolvement',
 'joblevel',
 'jobrole',
 'jobsatisfaction',
 'maritalstatus',
 'monthlyincome',
 'monthlyrate',
 'numcompaniesworked',
 'overtime',
 'percentsalaryhike',
 'performancerating',
 'relationshipsatisfaction',
 'stockoptionlevel',
 'totalworkingyears',
 'trainingtimeslastyear',
 'worklifebalance',
 'yearsatcompany',
 'yearsincurrentrole',
 'yearssincelastpromotion',
 'yearswithcurrmanager',
 'attrition_flag',
 'age_group',
 'tenure_group']

In [11]:
import sqlite3

conn = sqlite3.connect("../data/attrition.db")
df.to_sql("employees", conn, if_exists="replace", index=False)
pd.read_sql("SELECT COUNT(*) AS rows_loaded, SUM(attrition_flag) AS leavers FROM employees", conn)

,rows_loaded,leavers
0,1470,237


In [12]:
print(df.shape)
df["attrition"].value_counts()

(1470, 35)


attrition
No     1233
Yes     237
Name: count, dtype: int64

In [13]:
def run_sql_file(path):
    text = open(path).read()
    if not text.strip():
        print("This file is empty. Paste the query and save with Ctrl+S.")
        return
    for q in text.split(";"):
        if q.strip():
            display(pd.read_sql(q, conn))

In [14]:
run_sql_file("../sql/01_attrition_overview.sql")

,total_employees,employees_left,attrition_rate_pct
0,1470,237,16.12


In [15]:
run_sql_file("../sql/02_attrition_bydepartment_role.sql")

,department,employees,left_count,attrition_rate_pct
0,Sales,446,92,20.63
1,Human Resources,63,12,19.05
2,Research & Development,961,133,13.84


,jobrole,employees,left_count,attrition_rate_pct
0,Sales Representative,83,33,39.76
1,Laboratory Technician,259,62,23.94
2,Human Resources,52,12,23.08
3,Sales Executive,326,57,17.48
4,Research Scientist,292,47,16.10
5,Manufacturing Director,145,10,6.90
6,Healthcare Representative,131,9,6.87
7,Manager,102,5,4.90
8,Research Director,80,2,2.50


In [ ]:
run_sql_file("../sql/03_attrition_by_age_tenure.sql")